[← Back to Overview](./00_overview.ipynb) | [← Previous: Survival Analysis](./02_survival_analysis.ipynb) | [Next: Gene Expression →](./04_gene_expression_analysis.ipynb)

# TCGA-BRCA PAM50 Molecular Subtype Analysis

Detailed analysis of PAM50 molecular subtypes for breast cancer classification.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

# Import XenaCohort for easy data loading
from oncolearn.api.xenabrowser import XenaCohortBuilder

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Build the cohort
builder = XenaCohortBuilder()
cohort = builder.build_cohort("BRCA")

## Load PAM50 Data

Using the `XenaCohort` class, we load clinical data which includes PAM50 subtype classification.

In [ ]:
# Load clinical data (includes PAM50 subtypes) using XenaCohort
clinical = cohort.clinical()

# Extract PAM50-related columns
pam50_cols = ['sample'] + [col for col in clinical.columns if 'PAM50' in col or 'pam50' in col.lower()]
pam50 = clinical[pam50_cols].copy() if pam50_cols else clinical.copy()

# Remove duplicates based on sample
if 'sample' in pam50.columns:
    pam50 = pam50.drop_duplicates(subset='sample')

print(f"PAM50 data shape: {pam50.shape}")
print(f"Number of samples: {len(pam50)}")
print(f"\nPAM50-related columns: {[col for col in pam50.columns if col != 'sample']}")

pam50.head(10)

PAM50 data shape: (141, 2)
Number of samples: 141

Columns: ['sample', 'PAM50']


,sample,PAM50
0,TCGA-A7-A13F-01A,LumB
1,TCGA-A7-A13F-01A,LumB
2,TCGA-A7-A13E-01A,Basal
3,TCGA-A7-A13E-01A,Basal
4,TCGA-BH-A0DP-01A,LumA
5,TCGA-BH-A0DP-01A,LumA
6,TCGA-A2-A0EW-01A,LumA
7,TCGA-A2-A0EW-01A,LumA
8,TCGA-A7-A0CH-01A,LumA
9,TCGA-A7-A0CH-01A,LumA


In [3]:
# Data info
print("\nData Types:")
print(pam50.dtypes)

print("\nMissing Data:")
for col in pam50.columns:
    missing = pam50[col].isna().sum()
    missing_pct = (missing / len(pam50)) * 100
    print(f"{col:30s}: {missing:5d} ({missing_pct:5.1f}%)")


Data Types:
sample    object
PAM50     object
dtype: object

Missing Data:
sample                        :     0 (  0.0%)
PAM50                         :     0 (  0.0%)


## Subtype Distribution

In [4]:
# Find subtype column
subtype_cols = [
    col for col in pam50.columns if 'subtype' in col.lower() or col == 'Call']
print(f"Potential subtype columns: {subtype_cols}")

if subtype_cols:
    subtype_col = subtype_cols[0]
    print(f"\nUsing column: {subtype_col}")

    # Subtype distribution
    subtype_counts = pam50[subtype_col].value_counts()
    print(f"\nPAM50 Subtype Distribution:")
    print(subtype_counts)
    print(f"\nTotal samples: {subtype_counts.sum()}")

    # Percentages
    print(f"\nPercentage Distribution:")
    for subtype, count in subtype_counts.items():
        pct = (count / subtype_counts.sum()) * 100
        print(f"{subtype:15s}: {count:4d} ({pct:5.1f}%)")

Potential subtype columns: []


In [5]:
# Visualize subtype distribution
if subtype_cols:
    # Define colors for each subtype
    color_map = {
        'Basal': '#E41A1C',
        'Her2': '#377EB8',
        'LumA': '#4DAF4A',
        'LumB': '#984EA3',
        'Normal': '#FF7F00'
    }

    # Map colors to actual subtypes in data
    colors = [color_map.get(s, '#999999') for s in subtype_counts.index]

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # Bar plot
    axes[0, 0].bar(range(len(subtype_counts)),
                   subtype_counts.values, color=colors)
    axes[0, 0].set_xticks(range(len(subtype_counts)))
    axes[0, 0].set_xticklabels(subtype_counts.index, rotation=45, ha='right')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].set_title('PAM50 Molecular Subtype Distribution (Bar Chart)')
    # Add count labels
    for i, v in enumerate(subtype_counts.values):
        axes[0, 0].text(i, v + 5, str(v), ha='center', fontweight='bold')

    # Pie chart
    axes[0, 1].pie(subtype_counts.values, labels=subtype_counts.index, autopct='%1.1f%%',
                   colors=colors, startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})
    axes[0, 1].set_title('PAM50 Molecular Subtype Proportions')

    # Horizontal bar plot
    axes[1, 0].barh(range(len(subtype_counts)),
                    subtype_counts.values, color=colors)
    axes[1, 0].set_yticks(range(len(subtype_counts)))
    axes[1, 0].set_yticklabels(subtype_counts.index)
    axes[1, 0].set_xlabel('Count')
    axes[1, 0].set_title('PAM50 Subtype Distribution (Horizontal)')
    axes[1, 0].invert_yaxis()
    # Add count labels
    for i, v in enumerate(subtype_counts.values):
        axes[1, 0].text(v + 5, i, str(v), va='center', fontweight='bold')

    # Donut chart
    wedges, texts, autotexts = axes[1, 1].pie(subtype_counts.values, labels=subtype_counts.index,
                                              autopct='%1.1f%%', colors=colors, startangle=90,
                                              textprops={'fontsize': 10})
    # Draw circle for donut
    centre_circle = plt.Circle((0, 0), 0.70, fc='white')
    axes[1, 1].add_artist(centre_circle)
    axes[1, 1].set_title('PAM50 Subtype Proportions (Donut)')

    plt.tight_layout()
    plt.show()

## Subtype Characteristics

Clinical and biological characteristics of each PAM50 subtype.

In [6]:
# Create summary table
if subtype_cols:
    print("\nPAM50 Subtype Characteristics:")
    print("\n" + "="*80)

    characteristics = {
        'Basal': 'Triple-negative, high grade, poor prognosis, younger patients',
        'Her2': 'HER2 amplified, aggressive, responsive to HER2-targeted therapy',
        'LumA': 'ER+, low proliferation, best prognosis, hormone therapy responsive',
        'LumB': 'ER+, high proliferation, intermediate prognosis',
        'Normal': 'Normal-like, may represent contamination or distinct subtype'
    }

    for subtype in subtype_counts.index:
        count = subtype_counts[subtype]
        pct = (count / subtype_counts.sum()) * 100
        char = characteristics.get(subtype, 'Characteristics not defined')

        print(f"\n{subtype.upper()}:")
        print(f"  Count: {count} ({pct:.1f}%)")
        print(f"  Characteristics: {char}")

    print("\n" + "="*80)

## Additional PAM50 Features

In [7]:
# Check for other columns (e.g., correlation scores, confidence)
other_cols = [col for col in pam50.columns if col not in subtype_cols]
print(f"Other PAM50 columns ({len(other_cols)}):")
for col in other_cols:
    print(f"  - {col}")

# If there are correlation/confidence scores
numeric_cols = pam50.select_dtypes(include=[np.number]).columns.tolist()
if numeric_cols:
    print(f"\nNumeric columns: {numeric_cols}")

    # Summary statistics for numeric columns
    print("\nNumeric column statistics:")
    print(pam50[numeric_cols].describe())

Other PAM50 columns (2):
  - sample
  - PAM50


In [8]:
# If there are correlation scores for each subtype
correlation_cols = [col for col in pam50.columns if any(
    s in col for s in ['Basal', 'Her2', 'LumA', 'LumB', 'Normal'])]

if correlation_cols:
    print(f"\nCorrelation score columns found: {len(correlation_cols)}")
    print(correlation_cols)

    # Visualize correlation scores
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Box plot of correlations
    pam50[correlation_cols].boxplot(ax=axes[0], patch_artist=True)
    axes[0].set_xlabel('Subtype')
    axes[0].set_ylabel('Correlation Score')
    axes[0].set_title('Distribution of PAM50 Correlation Scores by Subtype')
    axes[0].tick_params(axis='x', rotation=45)

    # Heatmap of sample correlations
    sample_subset = pam50[correlation_cols].iloc[:50]  # First 50 samples
    sns.heatmap(sample_subset.T, cmap='RdBu_r', center=0, ax=axes[1],
                xticklabels=False, cbar_kws={'label': 'Correlation'})
    axes[1].set_ylabel('Subtype')
    axes[1].set_xlabel('Samples (first 50)')
    axes[1].set_title('PAM50 Correlation Heatmap')

    plt.tight_layout()
    plt.show()

## Integration with Clinical Data

In [9]:
# Load clinical and survival data for integration
try:
    clinical = pd.read_csv(
        DATA_DIR / 'TCGA-BRCA.clinical.tsv', sep='\t', low_memory=False)
    survival = pd.read_csv(DATA_DIR / 'TCGA-BRCA.survival.tsv', sep='\t')

    print("Clinical and survival data loaded for integration")
    print(f"Clinical samples: {len(clinical)}")
    print(f"Survival samples: {len(survival)}")
    print(f"PAM50 samples: {len(pam50)}")

    # Try to merge based on sample IDs
    # This will depend on the structure of your data
    print("\nNote: Integration analysis can be performed by matching sample IDs")
    print("across datasets to analyze subtype-specific clinical outcomes.")

except Exception as e:
    print(f"Could not load clinical/survival data for integration: {e}")

Clinical and survival data loaded for integration
Clinical samples: 1255
Survival samples: 1232
PAM50 samples: 141

Note: Integration analysis can be performed by matching sample IDs
across datasets to analyze subtype-specific clinical outcomes.


## Summary

In [10]:
print("\n" + "="*60)
print("TCGA-BRCA PAM50 Molecular Subtype Summary")
print("="*60)
print(f"Total samples: {len(pam50)}")
if subtype_cols:
    print(f"Unique subtypes: {pam50[subtype_col].nunique()}")
    print(f"\nSubtype distribution:")
    for subtype, count in subtype_counts.items():
        pct = (count / subtype_counts.sum()) * 100
        print(f"  {subtype:12s}: {count:4d} ({pct:5.1f}%)")
print("="*60)


TCGA-BRCA PAM50 Molecular Subtype Summary
Total samples: 141
